In [171]:
import logging
import warnings
import time
import datetime
from dataclasses import dataclass
from typing import Tuple, List
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from category_encoders import CountEncoder
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import (
    cross_val_score,
    KFold,
    GroupKFold,
    StratifiedKFold,
    TimeSeriesSplit,
)
from sklearn.linear_model import (
    Lasso,
    LogisticRegression,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    roc_auc_score,
    mean_absolute_error,
    root_mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
)
import lightgbm as lgb
import scipy
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import optuna
from optuna.samplers import TPESampler

warnings.filterwarnings("ignore")
# shap.initjs()

#### 1. Download data from Don’tGetKicked competition.

kaggle competitions download -c DontGetKicked<br>
or direct download

##### 1.1 Read all the data

In [153]:
path: str = "data/DontGetKicked/{name}.csv"
train_df: pd.DataFrame = pd.read_csv(
    path.format(name="training"), parse_dates=["PurchDate"]
)
test_df: pd.DataFrame = pd.read_csv(path.format(name="test"), parse_dates=["PurchDate"])

##### 1.2 Drop columns with N/A > 50%

In [154]:
NA_threshold = 0.5
NA_percent = train_df.isnull().mean()
to_drop = NA_percent[NA_percent >= NA_threshold].index
train_df_wo_na = train_df.drop(columns=to_drop)

##### 1.3 Simple impute N/A for other columns and scale numeric

In [155]:
target = "IsBadBuy"
datetime_col = "PurchDate"
numeric = (
    train_df_wo_na.drop(target, axis=1)
    .select_dtypes(exclude=["object", "datetime"])
    .columns
)
categories = train_df_wo_na.select_dtypes(include="object").columns

numeric_pipeline = Pipeline(
    [("num_imp", SimpleImputer(strategy="median")), ("st_scaler", StandardScaler())]
)

ct = ColumnTransformer(
    transformers=[
        ("num_pipeline", numeric_pipeline, numeric),
        (
            "cat_imp",
            SimpleImputer(strategy="constant", fill_value="unknown"),
            categories,
        ),
    ],
    remainder="drop",
)
train_df_wo_na_trnsfrm = ct.fit_transform(train_df_wo_na)

train_df_wo_na2 = pd.DataFrame(train_df_wo_na_trnsfrm)
train_df_wo_na2.columns = numeric.to_list() + categories.to_list()
train_df_wo_na2[datetime_col] = train_df_wo_na[datetime_col]
train_df_wo_na2[target] = train_df_wo_na[target]

#### 2. Design the train/validation/test split. <br>Use the "PurchDate" field for the split, test must be later than validation, same for validation and train: train.PurchDate < valid.PurchDate < test.PurchDate. <br>Use the first 1/3 of dates for the train, the last 1/3 of dates for the test, and the middle 1/3 for the validation set. Don’t use the test dataset until the end!

In [156]:
# all funcs from ML3
def split_by_date(
    X: pd.DataFrame,
    date_split: datetime.datetime,
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    info: pd.Series = X.dtypes
    date_in_df = [pd.api.types.is_datetime64_ns_dtype(x) for x in info.values]
    is_date_in_df = any(date_in_df)
    if not is_date_in_df:
        raise ValueError("No date col in df")

    # finds first entrance of datetype col
    date_col = info.index[date_in_df.index(True)]
    # sorts initial df to make proper split
    X = X.sort_values(by=date_col)
    X_train = X.loc[X[date_col] <= date_split]
    X_test = X.loc[X[date_col] > date_split]

    time_condition = X_train[date_col].max() < X_test[date_col].max()
    if not time_condition:
        raise ValueError("Split date cant provide proper split")

    return X_train, X_test


def validation_split_by_date(
    X: pd.DataFrame,
    validation_date: datetime.datetime,
    test_date: datetime.datetime,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:

    X_train, X_test = split_by_date(X, date_split=test_date)
    X_train, X_valid = split_by_date(X_train, date_split=validation_date)
    return X_train, X_valid, X_test


def check_split(df_list: List[pd.DataFrame], df_initial: pd.DataFrame) -> None:
    final_set: set = set(df_initial.index)
    for df in df_list:
        final_set -= set(df.index)
    print(final_set == set())
    [print(round(len(x) / len(df_initial), 2), end=" ") for x in df_list]

In [157]:
validation_date = datetime.datetime(year=2009, month=9, day=13)
test_date = datetime.datetime(year=2010, month=5, day=13)
X_train, X_valid, X_test = validation_split_by_date(
    X=train_df_wo_na2, validation_date=validation_date, test_date=test_date
)
check_split(df_list=[X_train, X_valid, X_test], df_initial=train_df_wo_na2)

True
0.33 0.34 0.33 

#### 3. Use LabelEncoder or OneHotEncoder from sklearn to preprocess categorical variables. Be careful with data leakage (fit Encoder to training and apply to validation & test). Consider another coding approach if you encounter new categorical values in validation & test (not seen in training): https://contrib.scikit-learn.org/category_encoders/count.html

In [158]:
cat_to_onehot_enc: list = []
cat_to_count_enc: list = []
for category in categories:
    # if there is many unique values we cant use OHE
    # try with n = 50
    if X_train[category].value_counts().count() < 21:
        cat_to_onehot_enc.append(category)
    else:
        cat_to_count_enc.append(category)

In [164]:
one_hot_enc = OneHotEncoder(
    sparse_output=False,
    drop="first",
    feature_name_combiner="concat",
    handle_unknown="ignore",
)
cat_one_hot_enc = one_hot_enc.fit_transform(X_train[cat_to_onehot_enc])
cat_one_hot_enc = pd.DataFrame(
    cat_one_hot_enc, columns=one_hot_enc.get_feature_names_out()
)

count_enc = CountEncoder(normalize=True, min_group_size=10)
cat_count_enc = count_enc.fit_transform(X_train[cat_to_count_enc])

X_train_ = pd.concat(
    [
        X_train[numeric].reset_index(drop=True),
        cat_one_hot_enc.reset_index(drop=True),
        cat_count_enc.reset_index(drop=True),
        X_train[target].reset_index(drop=True)
    ],
    axis=1,
)

In [166]:
df_transformed = [pd.DataFrame(), pd.DataFrame()]

for num, df in enumerate([X_valid, X_test]):
    cat_one_hot_enc = one_hot_enc.transform(df[cat_to_onehot_enc])
    cat_one_hot_enc = pd.DataFrame(cat_one_hot_enc, columns=one_hot_enc.get_feature_names_out())
    cat_count_enc = count_enc.transform(df[cat_to_count_enc])
    df_transformed[num] = pd.concat(
        [
            df[numeric].reset_index(drop=True), 
            cat_one_hot_enc.reset_index(drop=True), 
            cat_count_enc.reset_index(drop=True),
            df[target].reset_index(drop=True)
        ], 
        axis=1
    )

X_valid_, X_test_ = df_transformed

In [167]:
y_train_ = X_train_["IsBadBuy"]
X_train_ = X_train_.drop("IsBadBuy", axis=1)
y_valid_ = X_valid_["IsBadBuy"]
X_valid_ = X_valid_.drop("IsBadBuy", axis=1)
y_test_ = X_test_["IsBadBuy"]
X_test_ = X_test_.drop("IsBadBuy", axis=1)

#### 4. Train LogisticRegression, GaussianNB, KNN from sklearn on the training dataset and check the quality of your algorithms on the validation dataset. <br>The dependent variable (IsBadBuy) is binary. Don't forget to normalize your datasets before training your models.

You must get at least 0.15 Gini score (the best of all three). Which algorithm performs better? And why?


In [182]:
for model in LogisticRegression(), GaussianNB(), KNeighborsClassifier():
    model.fit(X=X_train_, y=y_train_)
    y_pred = model.predict_proba(X=X_valid_)[:, 1]
    # using 2*ROC AUC - 1 approach
    print(2 * roc_auc_score(y_true=y_valid_, y_score=y_pred) - 1)

0.4240252681689174
0.4253938271536226
0.22506024430604388


#### 5. Implement Gini score calculation. <br>You can use the 2*ROC AUC - 1 approach, so you need to implement the ROC AUC calculation. <br>Check if your metric is approximately equal to abs(2*sklearn.metrics.roc_auc_score - 1).

In [ ]:
def roc_auc():
    pass


def gini_score():
    return 2 * roc_auc() - 1

#### 6. Implement your own versions of LogisticRegression, KNN and NaiveBayes classifiers. <br>For LogisticRegression compute gradients with respect to the loss and use stochastic gradient descent. Can you reproduce the results from step 4?

Guidance for this task: Your model must be represented by class with methods fit, predict (predict_proba with 0.5 threshold), predict_proba.<br>
For LR moder, compute the loss gradient with respect to parameters w and parameter b in the fit function.<br>
Use a simple SGD approach to estimate optimal values of parameters.

#### 7. Try to create non-linear features, for example:

fractions: feature1/feature2
<br/>groupby features: `df[‘categorical_feature’].map(df.groupby(‘categorical_feature’)[‘continious_feature’].mean())`
<br/><br/>
Add new features to your pipeline, repeat step 4. Did you manage to increase your Gini score (you should!)?

#### 8. Determine the best features for the problem using the coefficients of the logistic model. <br>Try to eliminate useless features by hand and by L1 regularization. Which approach is better in terms of Gini score?

#### 9. Select your best model (algorithm + feature set) and tweak its hyperparameters to increase the Gini score on the validation dataset. <br>Which hyperparameters have the most impact?

#### 10. Check the Gini scores on all three datasets for your best model: training Gini, valid Gini, test Gini. Do you see a drop in performance when comparing the valid quality to the test quality? Is your model overfitted or not? Explain.

#### 11. Implement calculation of Recall, Precision, F1 score and AUC PR metrics. <br>Compare your algorithms on the test dataset using AUC PR metric.

#### 12. Which hard label metric do you prefer for the task of detecting "lemon" cars?